In [12]:
import numpy 
import pandas as pd
from pathlib import Path

# root_dir = Path("/media/volume/satclip-datasets/Datasets/ts-satclip-s2-1M")
root_dir = Path("/mnt/DATA/leca5365/landsat45-1M")
idx_df = pd.read_csv(root_dir / "full-index.csv")
idx_df["ts"] =  pd.to_datetime(idx_df["ts"], format='ISO8601')

print(idx_df.head())

                                               fn  \
0  1982/patch_LT04_L2SP_005024_19821128_02_T1.tif   
1  1982/patch_LT04_L2SP_005036_19820824_02_T2.tif   
2  1982/patch_LT04_L2SP_005038_19820824_02_T2.tif   
3  1982/patch_LT04_L2SP_006025_19821205_02_T1.tif   
4  1982/patch_LT04_L2SP_007029_19820822_02_T1.tif   

                                id        lat        lon  \
0  LT04_L2SP_005024_19821128_02_T1  51.484432 -56.837290   
1  LT04_L2SP_005036_19820824_02_T2  35.310551 -62.466621   
2  LT04_L2SP_005038_19820824_02_T2  31.385950 -62.779487   
3  LT04_L2SP_006025_19821205_02_T1  50.299022 -57.562771   
4  LT04_L2SP_007029_19820822_02_T1  44.863337 -62.862502   

                                ts  
0 1982-11-28 14:09:33.638044+00:00  
1 1982-08-24 14:12:26.133006+00:00  
2 1982-08-24 14:13:13.783025+00:00  
3 1982-12-05 14:16:11.124063+00:00  
4 1982-08-22 14:21:55.388031+00:00  


In [19]:
# Show how many samples per year
idx_df.groupby(idx_df['ts'].dt.year).size()

ts
1982      296
1983      307
1984    20149
1985    19996
1986    20216
1987    24894
1988    31568
1989    31078
1990    31103
1991    32456
1992    38388
1993    36022
1994    37086
1995    37112
1996    37000
1997    35818
1998    37624
1999    32742
2000    25662
2001    26057
2002    20426
2003    21688
2004    26467
2005    24906
2006    27935
2007    24291
2008    22674
2009    32212
2010    24042
2011    20952
2012        9
dtype: int64

In [28]:
# Create a balanced split
dts = pd.to_datetime(idx_df["ts"], format='ISO8601')
years = dts.dt.year.unique()

balanced = pd.DataFrame()

num_per_year = 20_000

for year in years:
    idx_df_year = idx_df[dts.dt.year == year]
    samples = idx_df_year.sample(n=min(num_per_year, len(idx_df_year)))
    balanced = pd.concat([balanced, samples], ignore_index=True)

balanced.to_csv(root_dir / "index-balanced-500k.csv", index=False)

# # Create mini splits
# balanced.sample(n=10_000).to_csv(root_dir / "index-10k.csv", index=False)
# balanced.sample(n=100_000).to_csv(root_dir / "index-100k.csv", index=False)

In [29]:
print("Number of samples: ", len(balanced))
print(balanced.groupby(balanced['ts'].dt.year).size())

Number of samples:  560608
ts
1982      296
1983      307
1984    20000
1985    19996
1986    20000
1987    20000
1988    20000
1989    20000
1990    20000
1991    20000
1992    20000
1993    20000
1994    20000
1995    20000
1996    20000
1997    20000
1998    20000
1999    20000
2000    20000
2001    20000
2002    20000
2003    20000
2004    20000
2005    20000
2006    20000
2007    20000
2008    20000
2009    20000
2010    20000
2011    20000
2012        9
dtype: int64


In [30]:
# Create mini splits, ensure that the smaller splits are subsets of the larger splits
balanced400k = balanced.sample(n=400_000)
balanced300k = balanced400k.sample(n=300_000)
balanced200k = balanced300k.sample(n=200_000)
balanced100k = balanced200k.sample(n=100_000)

balanced100k.to_csv(root_dir / "index-balanced-100k.csv", index=False)
balanced200k.to_csv(root_dir / "index-balanced-200k.csv", index=False)
balanced300k.to_csv(root_dir / "index-balanced-300k.csv", index=False)
balanced400k.to_csv(root_dir / "index-balanced-400k.csv", index=False)